# 🦠 COVID-19 Big Data Analytics — Full Analysis Notebook

**Big Data Analytics · TA-1 · Topic 10: COVID-19 Tracking & Prediction**

| Name | Roll No |
|------|---------|
| Arjun Jaiswal | 28 |
| Armaan Ahmed | 29 |
| Arnav Milmile | 30 |

---

This notebook walks through the **entire Big Data pipeline** for COVID-19 analysis:
1. Data Fetching (API Pipeline)
2. Data Cleaning & Preprocessing
3. Exploratory Data Analysis (EDA)
4. Visualization
5. Machine Learning — Case Prediction
6. Results & Interpretation


## 📦 Step 0: Install Dependencies

In [ ]:
# Run this cell once to install all required libraries
# !pip install pandas numpy matplotlib seaborn plotly scikit-learn streamlit requests

## 📡 Step 1: Data Fetching — Simulating an ETL Pipeline

In the real-world COVID-19 tracking systems, data flows through automated ETL pipelines on AWS/GCP/Azure.
Here we replicate this by fetching data from the **Our World in Data** public API.


In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('dark_background')
sns.set_palette('husl')

print('✅ Libraries imported successfully')

In [ ]:
from fetch_data import fetch_covid_data

# Fetch data (downloads from OWID or uses cached file)
df_raw = fetch_covid_data(use_cache=True)

print(f'\n📊 Dataset Shape: {df_raw.shape}')
print(f'📅 Date Range: {df_raw["date"].min()} → {df_raw["date"].max()}')
print(f'🌍 Countries: {df_raw["location"].unique().tolist()}')
df_raw.head()

## 🧹 Step 2: Data Cleaning & Preprocessing

Raw data always has issues — missing values, inconsistent formats, outliers.
This step mimics what **Apache Hadoop MapReduce** jobs do in production.


In [ ]:
from analyze import clean_data, get_summary_stats, get_monthly_aggregates

df = clean_data(df_raw)

print('📋 Missing Values After Cleaning:')
print(df.isnull().sum())

print('\n📊 Data Types:')
print(df.dtypes)

## 📊 Step 3: Exploratory Data Analysis (EDA)


In [ ]:
# Summary statistics per country
summary = get_summary_stats(df)
print('📋 Summary Statistics by Country:')
summary

In [ ]:
# Plot: Daily new cases for all countries
fig, ax = plt.subplots(figsize=(14, 6))

colors = ['#FF6B35', '#4CC9F0', '#7BC67E', '#F72585', '#FFD60A', '#9B5DE5']

for i, country in enumerate(df['location'].unique()):
    cdf = df[df['location'] == country].copy()
    rolling = cdf['new_cases'].rolling(7).mean()
    ax.plot(cdf['date'], rolling, label=country, color=colors[i % len(colors)], linewidth=2)

ax.set_title('Daily New Cases (7-Day Rolling Average)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('New Cases')
ax.legend(loc='upper left')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('../assets/daily_cases_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('📁 Chart saved to assets/')

In [ ]:
# Plot: Total cases comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart — total cases
summary_sorted = summary.sort_values('total_cases', ascending=True)
axes[0].barh(summary_sorted['location'], summary_sorted['total_cases'], color=colors[:len(summary_sorted)])
axes[0].set_title('Total Cases by Country', fontweight='bold')
axes[0].set_xlabel('Total Cases')

# Bar chart — vaccination rate
summary_sorted2 = summary.sort_values('max_vaccinated_pct', ascending=True)
axes[1].barh(summary_sorted2['location'], summary_sorted2['max_vaccinated_pct'], color=colors[:len(summary_sorted2)])
axes[1].set_title('Max Vaccination Rate (%)', fontweight='bold')
axes[1].set_xlabel('Vaccinated %')

plt.tight_layout()
plt.savefig('../assets/country_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap — monthly cases
monthly = get_monthly_aggregates(df)
pivot = monthly.pivot_table(index='location', columns='year_month', values='monthly_cases', aggfunc='sum')
pivot = pivot.fillna(0)

fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(pivot, cmap='Reds', ax=ax, linewidths=0.3, linecolor='#333')
ax.set_title('Monthly COVID-19 Cases Heatmap', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Country')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../assets/monthly_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 🤖 Step 4: Machine Learning — 30-Day Case Prediction

We train a **Linear Regression** model on historical daily case data.
This represents the concept of LSTM / Prophet time-series models in the case study.


In [ ]:
from predict import run_prediction_pipeline

# Run prediction for India
result = run_prediction_pipeline(df, country='India')

print('\n📊 Model Performance Metrics:')
for k, v in result['metrics'].items():
    print(f'   {k}: {v}')

In [ ]:
# Plot: Prediction vs Actual
cdf = result['country_df']
y_test = result['y_test']
y_pred = result['y_pred']
future_preds = result['future_preds']

last_date = cdf['date'].max()
future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=30)
test_dates = cdf['date'].iloc[-len(y_test):]

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(cdf['date'], cdf['rolling_avg'], color='#4CC9F0', linewidth=2, label='Actual (7-day avg)')
ax.plot(test_dates, y_pred, color='#ff7b72', linewidth=2, linestyle='--', label='Model Fit')
ax.plot(future_dates, future_preds, color='#bc8cff', linewidth=3, linestyle='--', label='30-Day Forecast')
ax.fill_between(future_dates, future_preds * 0.85, future_preds * 1.15, alpha=0.2, color='#bc8cff')
ax.axvline(last_date, color='#8b949e', linestyle=':', linewidth=1.5, label='Forecast Start')

ax.set_title('India — COVID-19 Case Prediction (30 Days Ahead)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Daily New Cases')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('../assets/ml_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

## ✅ Step 5: Conclusion

### Key Findings:
- The COVID-19 pandemic produced **terabytes of data daily** from health registries, labs, mobility reports, and social media
- Big Data tools (Hadoop, Spark) enabled processing of this data at scale
- ML models could **predict surges 2–4 weeks ahead**, allowing early resource allocation
- Vaccination data shows a clear inverse relationship with death rates in later waves

### Big Data Concepts Demonstrated:
| Concept | How We Used It |
|---------|---------------|
| Volume | Millions of records across 200+ countries |
| Velocity | Real-time API data fetching |
| Variety | Cases + Deaths + Vaccinations + Mobility |
| ETL Pipeline | `fetch_data.py` → `analyze.py` → `visualize.py` |
| ML | Linear Regression for time-series forecasting |
| Dashboard | Interactive Streamlit app |
